# 04 FeatureEngineering

Notebook นี้ใช้สร้างคอลัมน์เวลาเป้าหมายจากไฟล์ clean

In [25]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parents[0]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [26]:
import pandas as pd

from src.utils.paths import INTERIM_DATA_DIR

INTERIM_DATA_DIR

WindowsPath('D:/ML/data/interim')

## Load Clean Data

ใช้ไฟล์ `vw_timestamp_dashboard_clean.csv` ที่ผ่านขั้น clean มาแล้ว

In [27]:
source_path = INTERIM_DATA_DIR / 'vw_timestamp_dashboard_clean.csv'
df = pd.read_csv(source_path, encoding='utf-8-sig')

print(f'source: {source_path}')
print(f'shape: {df.shape}')
df.head()

source: D:\ML\data\interim\vw_timestamp_dashboard_clean.csv
shape: (7617, 37)


,PlantName,PickListType,PickDate,TruckSeqNo,CarType,CarNo,PackListNo,CustomerName,QueueTime,PrepareForward,...,PRESTIGEFittingSapAmount,NEUSTILEFittingSapAmount,DURAFittingSapAmount,ACCESSORIESSapAmount,TileStart,TileEnd,FittingStart,FittingEnd,AccStart,AccEnd
0,SB1,Walk in,2026-04-10 08:52:23,1,4 ล้อ,84-5388,SB1PL260410015,หจก.สำรวยเซรามิค,2026-04-10 08:52:32,N,...,0,0,0,0,2026-04-10 09:23:51,2026-04-10 09:24:56,2026-04-10 09:00:14,2026-04-10 09:04:32,NaN,NaN
1,SB1,Walk in,2026-04-10 08:48:55,4,6 ล้อ,70-4573,SB1PL260410014,บริษัท วันโฮม วัสดุก่อสร้าง จำกัด,2026-04-10 08:49:00,N,...,0,0,0,0,2026-04-10 09:08:03,2026-04-10 09:13:52,NaN,NaN,NaN,NaN
2,SB1,Walk in,2026-04-10 08:47:25,1,4 ล้อ,3ฒอ9423,SB1PL260410013,บ.รวมซีเมนต์99 จก.,2026-04-10 08:47:34,N,...,50,0,0,1,2026-04-10 09:15:01,2026-04-10 09:22:14,2026-04-10 08:55:17,2026-04-10 08:58:33,2026-04-10 08:50:45,2026-04-10 08:52:05
3,SB1,Walk in,2026-04-10 08:42:51,2,4 ล้อ,บล4874,SB1PL260410012,บ.เงินทองมาวัสดุ จก.,2026-04-10 08:42:56,N,...,30,0,0,0,2026-04-10 08:55:58,2026-04-10 08:56:39,2026-04-10 08:52:42,2026-04-10 08:54:33,NaN,NaN
4,SB1,Walk in,2026-04-10 08:29:57,6,6 ล้อ,71-0658,SB1PL260410011,บริษัท วันโฮม วัสดุก่อสร้าง จำกัด,2026-04-10 08:30:02,N,...,0,0,0,0,2026-04-10 08:45:24,2026-04-10 08:46:40,NaN,NaN,NaN,NaN


## Convert Timestamp Columns

แปลงคอลัมน์เวลาที่ใช้คำนวณให้เป็น `datetime` ก่อน

In [28]:
timestamp_columns = [
    'OperatorCarConfirm',
    'CarConfirm',
    'FirstPostPallet',
    'LastPostPallet',
    'PostingTime',
]

df_feature = df.copy()

for col in timestamp_columns:
    df_feature[col] = pd.to_datetime(df_feature[col], errors='coerce')

df_feature[timestamp_columns].dtypes

OperatorCarConfirm    datetime64[us]
CarConfirm            datetime64[us]
FirstPostPallet       datetime64[us]
LastPostPallet        datetime64[us]
PostingTime           datetime64[us]
dtype: object

## Create Time Columns

สร้างคอลัมน์ตามสูตรที่ต้องการ

In [29]:
df_feature['wait_call_min'] = (
    df_feature['CarConfirm'] - df_feature['OperatorCarConfirm']
).dt.total_seconds() / 60

df_feature['prepare_loading_min'] = (
    df_feature['FirstPostPallet'] - df_feature['CarConfirm']
).dt.total_seconds() / 60

df_feature['loading_time_min'] = (
    df_feature['LastPostPallet'] - df_feature['FirstPostPallet']
).dt.total_seconds() / 60

df_feature['close_job_min'] = (
    df_feature['PostingTime'] - df_feature['LastPostPallet']
).dt.total_seconds() / 60

df_feature['total_time_min'] = (
    df_feature['wait_call_min']
    + df_feature['prepare_loading_min']
    + df_feature['loading_time_min']
    + df_feature['close_job_min']
)

time_columns = [
    'wait_call_min',
    'prepare_loading_min',
    'loading_time_min',
    'close_job_min',
    'total_time_min',
]

df_feature[time_columns] = df_feature[time_columns].round(2)

df_feature[[
    'wait_call_min',
    'prepare_loading_min',
    'loading_time_min',
    'close_job_min',
    'total_time_min',
]].head()

,wait_call_min,prepare_loading_min,loading_time_min,close_job_min,total_time_min
0,15.63,1.40,12.02,24.33,53.38
1,7.95,2.72,6.23,10.80,27.70
2,9.62,13.63,7.10,19.32,49.67
3,1.58,10.43,15.42,8.12,35.55
4,5.40,4.67,9.88,13.87,33.82


## Check Result

ดูสถิติเบื้องต้นของคอลัมน์เวลาที่สร้างขึ้น

In [30]:
target_columns = [
    'wait_call_min',
    'prepare_loading_min',
    'loading_time_min',
    'close_job_min',
    'total_time_min',
]

df_feature[target_columns].describe().T

,count,mean,std,min,25%,50%,75%,max
wait_call_min,7617.0,9.203220,10.164076,-47.38,2.78,6.05,12.20,88.73
prepare_loading_min,7617.0,10.453856,11.738495,0.10,2.55,6.70,14.57,301.08
loading_time_min,7617.0,27.243150,20.725520,0.02,11.83,22.30,37.07,213.98
close_job_min,7617.0,15.661766,16.982628,1.18,7.03,11.68,19.37,339.90
total_time_min,7617.0,62.561926,28.098626,2.55,43.65,58.67,78.13,385.07


In [31]:
df_feature[[
    'OperatorCarConfirm',
    'CarConfirm',
    'FirstPostPallet',
    'LastPostPallet',
    'PostingTime',
    'wait_call_min',
    'prepare_loading_min',
    'loading_time_min',
    'close_job_min',
    'total_time_min',
]].head(10)

,OperatorCarConfirm,CarConfirm,FirstPostPallet,LastPostPallet,PostingTime,wait_call_min,prepare_loading_min,loading_time_min,close_job_min,total_time_min
0,2026-04-10 08:52:28,2026-04-10 09:08:06,2026-04-10 09:09:30,2026-04-10 09:21:31,2026-04-10 09:45:51,15.63,1.40,12.02,24.33,53.38
1,2026-04-10 08:48:56,2026-04-10 08:56:53,2026-04-10 08:59:36,2026-04-10 09:05:50,2026-04-10 09:16:38,7.95,2.72,6.23,10.80,27.70
2,2026-04-10 08:47:31,2026-04-10 08:57:08,2026-04-10 09:10:46,2026-04-10 09:17:52,2026-04-10 09:37:11,9.62,13.63,7.10,19.32,49.67
3,2026-04-10 08:42:54,2026-04-10 08:44:29,2026-04-10 08:54:55,2026-04-10 09:10:20,2026-04-10 09:18:27,1.58,10.43,15.42,8.12,35.55
4,2026-04-10 08:29:59,2026-04-10 08:35:23,2026-04-10 08:40:03,2026-04-10 08:49:56,2026-04-10 09:03:48,5.40,4.67,9.88,13.87,33.82
5,2026-04-10 08:18:45,2026-04-10 08:34:02,2026-04-10 08:36:06,2026-04-10 08:46:44,2026-04-10 09:02:12,15.28,2.07,10.63,15.47,43.45
6,2026-04-10 08:13:24,2026-04-10 08:36:00,2026-04-10 08:37:28,2026-04-10 08:58:18,2026-04-10 09:31:07,22.60,1.47,20.83,32.82,77.72
7,2026-04-10 07:58:25,2026-04-10 07:59:45,2026-04-10 08:03:30,2026-04-10 08:04:23,2026-04-10 08:15:08,1.33,3.75,0.88,10.75,16.72
8,2026-04-10 07:44:48,2026-04-10 07:52:06,2026-04-10 08:08:27,2026-04-10 08:30:41,2026-04-10 09:05:24,7.30,16.35,22.23,34.72,80.60
9,2026-04-10 07:35:54,2026-04-10 07:52:57,2026-04-10 08:04:12,2026-04-10 08:17:39,2026-04-10 08:20:13,17.05,11.25,13.45,2.57,44.32


## Save Feature Data

บันทึกไฟล์ที่เพิ่มคอลัมน์เวลาแล้วไว้ใน `data/interim`

In [32]:
output_path = INTERIM_DATA_DIR / 'vw_timestamp_dashboard_featured.csv'
df_feature.to_csv(output_path, index=False, encoding='utf-8-sig')
output_path

WindowsPath('D:/ML/data/interim/vw_timestamp_dashboard_featured.csv')